# mfs-agent-security: benchmark run on Kaggle

**Before running:** in the right panel, set **Accelerator: GPU T4 x2** and turn **Internet on**.

This version reruns **Qwen2.5-32B-Instruct** (4-bit AWQ, no Hugging Face account needed) with the fixed `provenance-consistent` defence, which now checks every SMS and invoice from the payee, not only those the agent read. It runs the 302 main cases and the 56 adaptive cases (`bench/cases_adaptive`, forged payee messages and invoices) with `provenance-consistent` only; the other defences were run before. Adaptive results go to `results_adaptive/`, so they never mix with the main results. Models already run are listed too, switched off with `#`; remove the `#` to run one again.

**Time:** about 1 to 1.5 hours. Downloading the model (about 19 GB) and starting it takes 15 to 20 minutes; each main-case run takes roughly 35 minutes and each adaptive run under 10 minutes. Use **Save Version → Save & Run All (Commit)** and let it run in the background. Results are saved after every run, so if Kaggle stops the session, the finished runs are kept in `results.zip`. If the model cannot start (for example, not enough GPU memory) or cannot call tools, the notebook says so, saves the server log in `results.zip`, and skips it.

In [ ]:
!nvidia-smi

## 1. Get the project and run its tests

In [ ]:
!git clone -q https://github.com/s4m404/mfs-agent-security.git
%cd mfs-agent-security
!pip install -q -r requirements.txt
!python -m pytest -q

## 2. Install vLLM

In [ ]:
!pip install -q vllm
# Kaggle's pre-installed torchaudio is built for a different CUDA version than the
# PyTorch that vLLM installs, which crashes vLLM at startup. Text models never use
# audio, so remove it.
!pip uninstall -y -q torchaudio
# Check that vLLM loads before starting any model; this stops here with the real error if not.
import subprocess
check = subprocess.run(["python", "-c", "import torch, vllm, vllm.entrypoints.cli.main; "
                        "print('vLLM', vllm.__version__, '| PyTorch', torch.__version__, '| CUDA', torch.version.cuda, "
                        "'| GPUs', torch.cuda.device_count())"], capture_output=True, text=True)
print(check.stdout or check.stderr[-3000:])
assert check.returncode == 0, "vLLM does not load. See the error above and vllm.log in the output."

## 3. Settings
Edit these lists to change what runs. Each model runs once per defence. Each model needs the vLLM *tool parser* that matches how it writes tool calls.

In [ ]:
MODELS = {  # model: vLLM tool parser
    "Qwen/Qwen2.5-32B-Instruct-AWQ": "hermes",            # larger model, 4-bit quantised to fit the T4s
    # "Qwen/Qwen2.5-7B-Instruct": "hermes",              # main run, done
    # "Qwen/Qwen2.5-14B-Instruct-AWQ": "hermes",         # main run, done
    # "Qwen/Qwen2.5-3B-Instruct": "hermes",              # main run, done
    # "NousResearch/Hermes-3-Llama-3.1-8B": "hermes",    # second family (Llama 3.1 8B), done
    # "ibm-granite/granite-3.3-8b-instruct": "granite",  # left out: tool calls came out as plain text
}
# Extra vLLM settings for models that need them. The 32B model only just fits
# the two T4s, so it may use more of the GPU memory and runs at most 8 cases at once.
EXTRA_ARGS = {
    "Qwen/Qwen2.5-32B-Instruct-AWQ": ["--gpu-memory-utilization", "0.95", "--max-num-seqs", "8"],
}
# Which defences to run on which case set: (cases folder, output folder, defences).
# The full first run of a new model uses ("bench/cases", "results", ["none", "provenance", "provenance-amount"]).
CASE_SETS = [
    ("bench/cases", "results", ["provenance-consistent"]),
    ("bench/cases_adaptive", "results_adaptive", ["provenance-consistent"]),
]
REPEATS = 1
WORKERS = 8   # cases run at the same time; set to 1 to run one by one

## 4. Helpers: start and stop the model server

In [ ]:
import subprocess, time, urllib.request, json, glob, os, signal
from openai import OpenAI

def start_server(model, parser, extra=()):
    proc = subprocess.Popen(
        ["vllm", "serve", model,
         "--dtype", "half",                 # T4 GPUs do not support bfloat16
         "--tensor-parallel-size", "2",
         "--max-model-len", "8192",
         "--enable-auto-tool-choice", "--tool-call-parser", parser,
         "--port", "8000", *extra],
        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for i in range(180):  # up to 30 minutes (large models take longer to download)
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            print(f"{model}: server ready after about {i * 10} seconds")
            return proc
        except Exception:
            if proc.poll() is not None:
                break
            time.sleep(10)
    print(f"{model}: server did not start. Last lines of vllm.log:")
    print(open("vllm.log").read()[-3000:])
    os.makedirs("results", exist_ok=True)  # keep the log in results.zip so the cause can be checked later
    with open(f"results/vllm_failed_{model.replace('/', '_')}.log", "w") as f:
        f.write(open("vllm.log").read())
    stop_server(proc)
    return None

def stop_server(proc):
    if proc and proc.poll() is None:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        try:
            proc.wait(timeout=60)
        except Exception:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
    time.sleep(15)  # let the GPUs free their memory

def tool_call_check(model):
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
    resp = client.chat.completions.create(
        model=model, temperature=0,
        messages=[{"role": "user", "content": "What is my wallet balance?"}],
        tools=[{"type": "function", "function": {
            "name": "check_balance", "description": "Return the user's balance.",
            "parameters": {"type": "object", "properties": {}}}}])
    ok = bool(resp.choices[0].message.tool_calls)
    print(f"{model}: tool calling {'works' if ok else 'DOES NOT WORK, skipping this model'}")
    return ok

## 5. Run everything
Each model: start server, check tool calling, run every defence, stop server.

In [ ]:
for model, parser in MODELS.items():
    print(f"\n########## {model} ##########")
    proc = start_server(model, parser, EXTRA_ARGS.get(model, ()))
    if proc is None:
        continue
    try:
        if not tool_call_check(model):
            continue
        for cases, out, defences in CASE_SETS:
            for defence in defences:
                print(f"\n===== {model} | {cases} | defence={defence} =====")
                !python scripts/run_bench.py --model {model} --base-url http://localhost:8000/v1 --cases {cases} --out {out} --defence {defence} --repeats {REPEATS} --workers {WORKERS} 2>&1 | grep -v "HTTP Request" | tail -25
                !zip -qr /kaggle/working/results.zip results results_adaptive
    finally:
        stop_server(proc)

## 6. Summary of all runs
`results.zip` in the Output panel has everything. Download it, unpack it next to the repository, and add the run folders to the `kaggle-results` branch (one dated folder per Kaggle run).

In [ ]:
!zip -qr /kaggle/working/results.zip results results_adaptive
for p in sorted(glob.glob("results*/*/summary.md")):
    print(p)
    print(open(p).read().split("## Attack success by injection language")[0])